In [1]:
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 22.8 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
import torch

In [3]:
'''import pandas as pd 

df = pd.read_csv('/kaggle/input/brainwave-dataset-prep/brainwave_preporcessed.csv')
# Map the class names to numerical values
def label_mapping(x):
    a = {'NEUTRAL': 0, 'NEGATIVE': 1, 'POSITIVE': 2}

    return a[x]

df['1516'] = df['1516'].apply(label_mapping)

# Display the updated DataFrame
print("Updated DataFrame:")
df.to_csv("brainwave_preporcessed.csv", index=False)'''

'import pandas as pd \n\ndf = pd.read_csv(\'/kaggle/input/brainwave-dataset-prep/brainwave_preporcessed.csv\')\n# Map the class names to numerical values\ndef label_mapping(x):\n    a = {\'NEUTRAL\': 0, \'NEGATIVE\': 1, \'POSITIVE\': 2}\n\n    return a[x]\n\ndf[\'1516\'] = df[\'1516\'].apply(label_mapping)\n\n# Display the updated DataFrame\nprint("Updated DataFrame:")\ndf.to_csv("brainwave_preporcessed.csv", index=False)'

In [4]:
'''import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# # --- Data Loading and Graph Construction ---
# df = pd.read_csv("/kaggle/working/brainwave_preporcessed.csv")

# # Select channel columns that are named with digits only (e.g., "0", "1", ..., "1515")
# channel_cols = [col for col in df.columns if col.isdigit()]
# if not channel_cols:
#     raise ValueError("No channel columns found in the CSV. Please verify your column names.")

# num_channels = len(channel_cols)

import torch
import pandas as pd
from torch_geometric.data import Data

# Load the dataset
df = pd.read_csv("/kaggle/input/brainwave-final-prep/brainwave_preporcessed (2).csv")  # Adjust the path accordingly

num_features = 1516  # Total feature columns
num_channels = 4    # EEG has 4 channels
features_per_channel = num_features // num_channels  # Features per channel

# Generate edges: Intra-channel (within same EEG channel)
edge_list = []
for ch in range(num_channels):
    start_idx = ch * features_per_channel
    end_idx = start_idx + features_per_channel

    # Fully connect nodes within each EEG channel (excluding self-loops)
    for i in range(start_idx, end_idx):
        for j in range(start_idx, end_idx):
            if i != j:
                edge_list.append([i, j])

# Generate edges: Inter-channel (corresponding feature nodes across channels)
for f in range(features_per_channel):
    for ch1 in range(num_channels):
        for ch2 in range(ch1 + 1, num_channels):
            edge_list.append([ch1 * features_per_channel + f, ch2 * features_per_channel + f])
            edge_list.append([ch2 * features_per_channel + f, ch1 * features_per_channel + f])  # Undirected

# Convert edges to PyTorch format
edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()

# Create PyTorch Geometric Data objects for each EEG sample
data_list = []
for idx, row in df.iterrows():
    # Extract EEG feature values and reshape as a node feature matrix
    x = torch.tensor(row.iloc[:num_features].values, dtype=torch.float).view(-1, 1)  # (num_nodes, 1)
    
    # Extract the target label
    y = torch.tensor([row.iloc[-1]], dtype=torch.long)  # Assuming last column is the label
    
    # Construct graph object
    data = Data(x=x, edge_index=edge_index, y=y)
    data_list.append(data)

print(f"✅ Constructed {len(data_list)} EEG Graphs with {num_features} nodes each.")


# Split the data into training and testing sets
train_data, test_data = train_test_split(data_list, test_size=0.2, random_state=42)
train_loader = DataLoader(train_data, batch_size=8, shuffle=True)
test_loader = DataLoader(test_data, batch_size=8, shuffle=False)

import torch
import torch.nn.functional as F
from torch.nn import Linear, BatchNorm1d
from torch_geometric.nn import ChebConv, GCNConv, global_mean_pool, global_max_pool

class HybridWaveletPooling(torch.nn.Module):
    """Hybrid pooling mechanism that combines mean, max, and wavelet-based adaptive pooling."""
    def __init__(self, in_channels, out_channels):
        super(HybridWaveletPooling, self).__init__()
        #self.linear = Linear(in_channels, out_channels)
    
    def forward(self, x, batch):
        mean_pooled = global_mean_pool(x, batch)  # Mean Pooling
        max_pooled = global_max_pool(x, batch)    # Max Pooling
        #wavelet_pooled = torch.tanh(self.linear(x))  # Wavelet-based Adaptive Pooling

        # Concatenating all three pooled features
        #print(mean_pooled.shape, max_pooled.shape, x.shape)
        pooled = torch.cat([mean_pooled, max_pooled], dim=1)
        return pooled


import torch
import torch.nn.functional as F
from torch.nn import Linear, BatchNorm1d
from torch_geometric.nn import GCNConv, ChebConv

class SpectralGraphWaveletGNN(torch.nn.Module):
    def __init__(self, hidden_channels, num_classes, K=3, dropout=0.3):
        super(SpectralGraphWaveletGNN, self).__init__()
        
        self.dropout = dropout

        # **Spatial View (GCN)**
        self.spatial_conv1 = GCNConv(1, hidden_channels)
        self.spatial_bn1 = BatchNorm1d(hidden_channels)
        self.spatial_conv2 = GCNConv(hidden_channels, hidden_channels)
        self.spatial_bn2 = BatchNorm1d(hidden_channels)
        
        # **Spectral View (Graph Wavelets - SGWT using ChebConv)**
        self.spectral_conv1 = ChebConv(1, hidden_channels, K=K)
        self.spectral_bn1 = BatchNorm1d(hidden_channels)
        self.spectral_conv2 = ChebConv(hidden_channels, hidden_channels, K=K)
        self.spectral_bn2 = BatchNorm1d(hidden_channels)
        
        # **Fusion Layer (Spatial + Spectral)**
        self.fusion_fc = Linear(2 * hidden_channels, hidden_channels)

        # **Hybrid Wavelet Pooling**
        self.hybrid_pool = HybridWaveletPooling(hidden_channels, hidden_channels)

        # **Final Classification**
        self.lin1 = Linear(2 * hidden_channels, hidden_channels // 2)  # Adapting for pooled dimensions
        self.lin2 = Linear(hidden_channels // 2, num_classes)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        
        # **Spatial View (GCN)**
        spatial_x = self.spatial_conv1(x, edge_index)
        spatial_x = self.spatial_bn1(spatial_x)
        spatial_x = F.gelu(spatial_x)
        spatial_x = self.spatial_conv2(spatial_x, edge_index)
        spatial_x = self.spatial_bn2(spatial_x)
        spatial_x = F.gelu(spatial_x)
        
        # **Spectral View (Graph Wavelet - SGWT)**
        spectral_x = self.spectral_conv1(x, edge_index)
        spectral_x = self.spectral_bn1(spectral_x)
        spectral_x = F.gelu(spectral_x)
        spectral_x = self.spectral_conv2(spectral_x, edge_index)
        spectral_x = self.spectral_bn2(spectral_x)
        spectral_x = F.gelu(spectral_x)
        
        # **Fusion of Spatial & Spectral Views**
        x = torch.cat([spatial_x, spectral_x], dim=1)  # Feature concatenation
        x = self.fusion_fc(x)
        x = F.gelu(x)
        
        # **Hybrid Wavelet Pooling**
        x = self.hybrid_pool(x, batch)
        
        # **Final Classification**
        x = self.lin1(x)
        x = F.gelu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)

        return x

# Define GNN Model
import torch
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing, GCNConv, global_mean_pool
from torch_geometric.utils import add_self_loops


class HaarWaveletConv(MessagePassing):
    def __init__(self, in_channels, out_channels):
        super(HaarWaveletConv, self).__init__(aggr='mean')
        self.lin = torch.nn.Linear(in_channels, out_channels)

    def forward(self, x, edge_index):
        edge_index, _ = add_self_loops(edge_index, num_nodes=x.size(0))
        x = self.lin(x)
        return self.propagate(edge_index, x=x)

    def message(self, x_j, x_i):
        return (x_j - x_i)/(2**0.5)

    def update(self, aggr_out):
        return aggr_out

class CustomGCNN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.5, sim_threshold=0.7):
        super(CustomGCNN, self).__init__()

        self.dropout = dropout
        self.sim_threshold = sim_threshold

        # Haar wavelet branch
        self.haar_conv1 = HaarWaveletConv(in_channels, hidden_channels)
        self.haar_conv2 = HaarWaveletConv(hidden_channels, hidden_channels)
        self.haar_conv3 = HaarWaveletConv(hidden_channels, hidden_channels)

        # Spatial GCN branch
        self.spatial_conv1 = GCNConv(in_channels, hidden_channels)
        self.spatial_conv2 = GCNConv(hidden_channels, hidden_channels)
        self.spatial_conv3 = GCNConv(hidden_channels, hidden_channels)

        # Fully connected classification layer
        self.fc = torch.nn.Linear(hidden_channels, out_channels)

    def nash_bargaining_fusion(self, x_haar, x_spatial):
        # Check similarity for win-win agreement
        sim = F.cosine_similarity(x_haar, x_spatial, dim=1)

        # Win-win mask: where similarity is high enough
        winwin_mask = sim > self.sim_threshold

        # Nash fusion (geometric mean weighted by log utility)
        fused = torch.sqrt(torch.clamp(x_haar * x_spatial, min=1e-6))

        # If not win-win, fallback to average
        fallback = 0.5 * (x_haar + x_spatial)

        return torch.where(winwin_mask.unsqueeze(1), fused, fallback)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        # Haar wavelet branch
        x_haar = self.haar_conv1(x, edge_index)
        x_haar = F.relu(x_haar)
        x_haar = self.haar_conv2(x_haar, edge_index)
        x_haar = F.relu(x_haar)
        #x_haar = F.dropout(x_haar, p=self.dropout, training=self.training)

        x_haar = self.haar_conv3(x_haar, edge_index)
        x_haar = F.relu(x_haar)
        #x_haar = F.dropout(x_haar, p=self.dropout, training=self.training)

        # Spatial GCN branch
        x_spatial = self.spatial_conv1(x, edge_index)
        x_spatial = F.relu(x_spatial)
        #x_spatial = F.dropout(x_spatial, p=self.dropout, training=self.training)

        x_spatial = self.spatial_conv2(x_spatial, edge_index)
        x_spatial = F.relu(x_spatial)
        x_spatial = self.spatial_conv3(x_spatial, edge_index)
        x_spatial = F.relu(x_spatial)
        #x_spatial = F.dropout(x_spatial, p=self.dropout, training=self.training)

        # Concatenate features
        x_combined = self.nash_bargaining_fusion(x_haar, x_spatial)

        # Global pooling
        x_pooled = global_mean_pool(x_combined, batch)

        # Final classification
        x_pooled = F.dropout(x_pooled, p=self.dropout, training=self.training)
        out = self.fc(x_pooled)

        return out



# Determine device and number of classes from labels
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_classes = len(np.unique(df['1516']))
import torch
import torch.nn.functional as F
from tqdm import tqdm

# Model Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#model = SpectralGraphWaveletGNN(hidden_channels=64, num_classes=num_classes).to(device)
model = CustomGCNN(in_channels=1, hidden_channels=64, out_channels=num_classes).to(device)
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Total trainable parameters:", count_parameters(model))

# Optimizer & Loss
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
criterion = torch.nn.CrossEntropyLoss()

# Learning Rate Scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True)

# Track best validation loss
best_val_loss = float("inf")
model_save_path = "best_model_BW.pth"

# --- Training & Evaluation Functions ---
def train_epoch(loader):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    loop = tqdm(loader, desc="Training", leave=False)
    for data in loop:
        data = data.to(device)
        optimizer.zero_grad()
        out = model(data)
        loss = criterion(out, data.y)
        loss.backward()
        
        # Gradient Clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        
        optimizer.step()
        
        batch_size = data.num_graphs
        total_loss += loss.item() * batch_size
        preds = out.argmax(dim=1)
        correct += int((preds == data.y).sum())
        total += batch_size
        
        # Update tqdm progress bar
        loop.set_postfix(loss=f"{loss.item():.4f}")
        
    return total_loss / total, correct / total

def evaluate_epoch(loader):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        loop = tqdm(loader, desc="Evaluating", leave=False)
        for data in loop:
            data = data.to(device)
            out = model(data)
            loss = criterion(out, data.y)
            
            batch_size = data.num_graphs
            total_loss += loss.item() * batch_size
            preds = out.argmax(dim=1)
            correct += int((preds == data.y).sum())
            total += batch_size
            
            # Update tqdm progress bar
            loop.set_postfix(loss=f"{loss.item():.4f}")
    
    return total_loss / total, correct / total

# --- Main Training Loop ---
num_epochs = 100
for epoch in range(1, num_epochs + 1):
    print(f"\nEpoch {epoch}/{num_epochs}")

    # Training
    train_loss, train_acc = train_epoch(train_loader)
    
    # Validation
    val_loss, val_acc = evaluate_epoch(test_loader)

    # Learning rate scheduling
    scheduler.step(val_loss)

    # Save the best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), model_save_path)
        print(f"✔ Model saved at epoch {epoch} with validation loss {val_loss:.4f}")

    # Epoch Summary
    print(f"📌 Epoch {epoch}: "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f} | "
          f"LR: {optimizer.param_groups[0]['lr']:.6f}")


def train_model(train_loader, val_loader, model, optimizer, criterion, epochs=1):
    global best_val_loss
    model.train()
    for epoch in range(epochs):
        total_train_loss, total_val_loss = 0, 0
        all_train_preds, all_train_labels = [], []
        
        # Training Loop
        for batch in tqdm(train_loader, desc=f"Training Epoch {epoch+1}"):
            batch = batch.to(device)
            optimizer.zero_grad()
            out = model(batch)
            loss = criterion(out, batch.y)
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()
            all_train_preds.extend(out.argmax(dim=1).cpu().numpy())
            all_train_labels.extend(batch.y.cpu().numpy())
        
        train_acc = accuracy_score(all_train_labels, all_train_preds)
        
        # Validation Loop
        model.eval()
        all_val_preds, all_val_labels = [], []
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Val Epoch {epoch+1}"):
                batch = batch.to(device)
                out = model(batch)
                loss = criterion(out, batch.y)
                total_val_loss += loss.item()
                all_val_preds.extend(out.argmax(dim=1).cpu().numpy())
                all_val_labels.extend(batch.y.cpu().numpy())
        
        val_acc = accuracy_score(all_val_labels, all_val_preds)
        avg_train_loss = total_train_loss / len(train_loader)
        avg_val_loss = total_val_loss / len(val_loader)
        scheduler.step(avg_val_loss)
        
        print(f"Epoch {epoch+1}: Train Loss = {avg_train_loss:.4f}, Train Acc = {train_acc:.4f}, Val Loss = {avg_val_loss:.4f}, Val Acc = {val_acc:.4f}")
        
        # Save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), "best_model.pth")
            print("Saved new best model!")

def test_model(test_loader, model):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Testing"):
            batch = batch.to(device)
            out = model(batch)
            preds = out.argmax(dim=1).cpu().numpy()
            labels = batch.y.cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels)
    
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted')
    precision = precision_score(all_labels, all_preds, average='weighted')
    recall = recall_score(all_labels, all_preds, average='weighted')
    
    print(f"Test Accuracy: {acc:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")

# Train and Test the model
train_model(train_loader, test_loader, model, optimizer, criterion, epochs=150)
test_model(test_loader, model)
'''


'import torch\nimport torch.nn.functional as F\nimport pandas as pd\nimport numpy as np\nfrom torch_geometric.data import Data, DataLoader\nfrom torch_geometric.nn import GCNConv, global_mean_pool\nfrom sklearn.model_selection import train_test_split\nfrom tqdm import tqdm\nfrom sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score\n\n# # --- Data Loading and Graph Construction ---\n# df = pd.read_csv("/kaggle/working/brainwave_preporcessed.csv")\n\n# # Select channel columns that are named with digits only (e.g., "0", "1", ..., "1515")\n# channel_cols = [col for col in df.columns if col.isdigit()]\n# if not channel_cols:\n#     raise ValueError("No channel columns found in the CSV. Please verify your column names.")\n\n# num_channels = len(channel_cols)\n\nimport torch\nimport pandas as pd\nfrom torch_geometric.data import Data\n\n# Load the dataset\ndf = pd.read_csv("/kaggle/input/brainwave-final-prep/brainwave_preporcessed (2).csv")  # Adjust the path accord

In [5]:
import torch
import torch_geometric
from torch.utils.data import Dataset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool
import numpy as np
import pywt
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import pandas as pd


concatenated_df = pd.read_csv("/kaggle/input/brainwave-final-prep/brainwave_preporcessed (2).csv")

class EEGGraphDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe
        self.num_channels = 8
        self.features_per_channel = 189
        self.labels = torch.tensor(dataframe['1516'].values, dtype=torch.long)

    def haar_wavelet_transform(self, x):
        coeffs = pywt.wavedec(x, 'haar', level=1)
        return torch.tensor(np.concatenate(coeffs), dtype=torch.float)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        features = []
        
        for i in range(self.num_channels):
            start_idx = i * self.features_per_channel
            end_idx = start_idx + self.features_per_channel
            channel_features = row[start_idx:end_idx].values.astype(np.float32)
            #transformed_features = self.haar_wavelet_transform(channel_features)
            features.append(channel_features)
        
        x = torch.tensor(features)  # Shape: (62, 5)
        edge_index = torch_geometric.utils.dense_to_sparse(torch.ones(self.num_channels, self.num_channels))[0]
        y = self.labels[index]
        
        return Data(x=x, edge_index=edge_index, y=y)
    
    def __len__(self):
        return len(self.dataframe)

# Load dataset
dataset = EEGGraphDataset(concatenated_df)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])
train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False)

# Define GNN Model
import torch
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing, GCNConv, global_mean_pool
from torch_geometric.utils import add_self_loops


class HaarWaveletConv(MessagePassing):
    def __init__(self, in_channels, out_channels):
        super(HaarWaveletConv, self).__init__(aggr='mean')
        self.lin = torch.nn.Linear(in_channels, out_channels)

    def forward(self, x, edge_index):
        edge_index, _ = add_self_loops(edge_index, num_nodes=x.size(0))
        x = self.lin(x)
        return self.propagate(edge_index, x=x)

    def message(self, x_j, x_i):
        return (x_j - x_i)/(2**0.5)

    def update(self, aggr_out):
        return aggr_out


class CustomGCNN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.5, sim_threshold=0.7):
        super(CustomGCNN, self).__init__()

        self.dropout = dropout
        self.sim_threshold = sim_threshold

        # Haar wavelet branch
        self.haar_conv1 = HaarWaveletConv(in_channels, hidden_channels)
        self.haar_conv2 = HaarWaveletConv(hidden_channels, hidden_channels)
        self.haar_conv3 = HaarWaveletConv(hidden_channels, hidden_channels)

        # Spatial GCN branch
        self.spatial_conv1 = GCNConv(in_channels, hidden_channels)
        self.spatial_conv2 = GCNConv(hidden_channels, hidden_channels)
        self.spatial_conv3 = GCNConv(hidden_channels, hidden_channels)

        # Fully connected classification layer
        self.fc = torch.nn.Linear(hidden_channels, out_channels)

    def nash_bargaining_fusion(self, x_haar, x_spatial):
        # Check similarity for win-win agreement
        sim = F.cosine_similarity(x_haar, x_spatial, dim=1)

        # Win-win mask: where similarity is high enough
        winwin_mask = sim > self.sim_threshold

        # Nash fusion (geometric mean weighted by log utility)
        fused = torch.sqrt(torch.clamp(x_haar * x_spatial, min=1e-6))

        # If not win-win, fallback to average
        fallback = 0.5 * (x_haar + x_spatial)

        return torch.where(winwin_mask.unsqueeze(1), fused, fallback)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        # Haar wavelet branch
        x_haar = self.haar_conv1(x, edge_index)
        x_haar = F.relu(x_haar)
        x_haar = self.haar_conv2(x_haar, edge_index)
        x_haar = F.relu(x_haar)
        #x_haar = F.dropout(x_haar, p=self.dropout, training=self.training)

        x_haar = self.haar_conv3(x_haar, edge_index)
        x_haar = F.relu(x_haar)
        #x_haar = F.dropout(x_haar, p=self.dropout, training=self.training)

        # Spatial GCN branch
        x_spatial = self.spatial_conv1(x, edge_index)
        x_spatial = F.relu(x_spatial)
        #x_spatial = F.dropout(x_spatial, p=self.dropout, training=self.training)

        x_spatial = self.spatial_conv2(x_spatial, edge_index)
        x_spatial = F.relu(x_spatial)
        x_spatial = self.spatial_conv3(x_spatial, edge_index)
        x_spatial = F.relu(x_spatial)
        #x_spatial = F.dropout(x_spatial, p=self.dropout, training=self.training)

        # Concatenate features
        x_combined = self.nash_bargaining_fusion(x_haar, x_spatial)

        # Global pooling
        x_pooled = global_mean_pool(x_combined, batch)

        # Final classification
        x_pooled = F.dropout(x_pooled, p=self.dropout, training=self.training)
        out = self.fc(x_pooled)

        return out

    

  

# Initialize model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CustomGCNN(in_channels=189, hidden_channels=32, out_channels=len(concatenated_df['1516'].unique())).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
criterion = torch.nn.CrossEntropyLoss()
best_val_loss = float("inf")

def train_model(train_loader, val_loader, model, optimizer, criterion, epochs=10):
    global best_val_loss
    model.train()
    for epoch in range(epochs):
        total_train_loss, total_val_loss = 0, 0
        all_train_preds, all_train_labels = [], []
        
        # Training Loop
        for batch in tqdm(train_loader, desc=f"Training Epoch {epoch+1}"):
            batch = batch.to(device)
            optimizer.zero_grad()
            out = model(batch)
            loss = criterion(out, batch.y)
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()
            all_train_preds.extend(out.argmax(dim=1).cpu().numpy())
            all_train_labels.extend(batch.y.cpu().numpy())
        
        train_acc = accuracy_score(all_train_labels, all_train_preds)
        
        # Validation Loop
        model.eval()
        all_val_preds, all_val_labels = [], []
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Val Epoch {epoch+1}"):
                batch = batch.to(device)
                out = model(batch)
                loss = criterion(out, batch.y)
                total_val_loss += loss.item()
                all_val_preds.extend(out.argmax(dim=1).cpu().numpy())
                all_val_labels.extend(batch.y.cpu().numpy())
        
        val_acc = accuracy_score(all_val_labels, all_val_preds)
        avg_train_loss = total_train_loss / len(train_loader)
        avg_val_loss = total_val_loss / len(val_loader)
        scheduler.step(avg_val_loss)
        
        print(f"Epoch {epoch+1}: Train Loss = {avg_train_loss:.4f}, Train Acc = {train_acc:.4f}, Val Loss = {avg_val_loss:.4f}, Val Acc = {val_acc:.4f}")
        
        # Save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), "best_model.pth")
            print("Saved new best model!")

def test_model(test_loader, model):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Testing"):
            batch = batch.to(device)
            out = model(batch)
            preds = out.argmax(dim=1).cpu().numpy()
            labels = batch.y.cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels)
    
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted')
    precision = precision_score(all_labels, all_preds, average='weighted')
    recall = recall_score(all_labels, all_preds, average='weighted')
    
    print(f"Test Accuracy: {acc:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")

# Train and Test the model
train_model(train_loader, test_loader, model, optimizer, criterion, epochs=150)
test_model(test_loader, model)

Training Epoch 1:   0%|          | 0/4 [00:00<?, ?it/s]<ipython-input-5-adaf10701b61>:39: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:278.)
  x = torch.tensor(features)  # Shape: (62, 5)
Val Epoch 1: 100%|██████████| 1/1 [00:00<00:00,  1.58it/s]


Epoch 1: Train Loss = 26.2987, Train Acc = 0.4094, Val Loss = 3.7929, Val Acc = 0.7330
Saved new best model!


Val Epoch 2: 100%|██████████| 1/1 [00:00<00:00,  1.59it/s]


Epoch 2: Train Loss = 2.6615, Train Acc = 0.6850, Val Loss = 2.4943, Val Acc = 0.7307
Saved new best model!


Val Epoch 3: 100%|██████████| 1/1 [00:00<00:00,  1.56it/s]


...

Epoch 149: Train Loss = 0.0572, Train Acc = 0.9853, Val Loss = 0.1737, Val Acc = 0.9485


Val Epoch 150: 100%|██████████| 1/1 [00:00<00:00,  1.58it/s]


Epoch 150: Train Loss = 0.0554, Train Acc = 0.9853, Val Loss = 0.1737, Val Acc = 0.9485


Testing: 100%|██████████| 1/1 [00:00<00:00,  1.61it/s]

Test Accuracy: 0.9485
F1 Score: 0.9485
Precision: 0.9487
Recall: 0.9485
